In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


DISEASE-AWARE ATTENTION

CELL 1 — IMPORT LIBRARIES AND DEFINE MODULE PATHS


This notebook implements the Disease-Aware Attention stage of the
QCDP-BiFormer architecture.

The previous stages produced:

    train_features.pt
        → Trained FiLM-conditioned image features [4491, 768]

    disease_prototypes.pt
        → Disease Prototype Memory [8, 768]

In this module, each image feature will attend to the disease prototype
memory.

Conceptually:

    Image Feature → Query

    Disease Prototypes → Keys + Values

The attention mechanism will produce a disease-aware representation that
combines information from the original image feature and the prototype
memory.

######The output of this module will later be used for bilateral reasoning and cross-eye attention.

---

In [2]:
# ============================================================
# CELL 1 — IMPORT LIBRARIES AND DEFINE MODULE PATHS
# ============================================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

ROOT = "/content/drive/My Drive/Eye Disease/Dataset"

TRAIN_FEATURES_PATH = os.path.join(
    ROOT,
    "train_features.pt"
)

PROTOTYPE_PATH = os.path.join(
    ROOT,
    "disease_prototypes.pt"
)

ATTENTION_SAVE_PATH = os.path.join(
    ROOT,
    "disease_aware_attention.pt"
)

LABEL_COLUMNS = [
    "N", "D", "G", "C",
    "A", "H", "M", "O"
]

NUM_CLASSES = len(LABEL_COLUMNS)
FEATURE_DIM = 768

print("Device:", device)
print("Feature dimension:", FEATURE_DIM)
print("Number of disease prototypes:", NUM_CLASSES)

Device: cuda
Feature dimension: 768
Number of disease prototypes: 8


CELL 2 — LOAD AND VERIFY FEATURE AND PROTOTYPE MEMORY


We now load the learned image features and the Disease Prototype Memory.

The image features represent:

    4,491 training eyes × 768 dimensions

The prototype memory represents:

    8 diseases × 768 dimensions

Before building attention, we verify that both representations share the
same feature dimension.

This is essential because the image features and disease prototypes must
exist in the same learned representation space before attention projections
are applied.

---

In [3]:
# ============================================================
# CELL 2 — LOAD AND VERIFY FEATURE AND PROTOTYPE MEMORY
# ============================================================

feature_data = torch.load(
    TRAIN_FEATURES_PATH,
    map_location="cpu",
    weights_only=False
)

prototype_memory = torch.load(
    PROTOTYPE_PATH,
    map_location="cpu",
    weights_only=False
)

train_features = feature_data["features"].float()
train_labels = feature_data["labels"].float()

disease_prototypes = (
    prototype_memory["disease_prototypes"]
    .float()
)

assert train_features.shape[1] == FEATURE_DIM
assert disease_prototypes.shape == (
    NUM_CLASSES,
    FEATURE_DIM
)

assert torch.isfinite(train_features).all()
assert torch.isfinite(disease_prototypes).all()

print(
    "Image features:",
    train_features.shape
)

print(
    "Disease prototypes:",
    disease_prototypes.shape
)

print(
    "\nSUCCESS: Feature and prototype memory "
    "are ready for Disease-Aware Attention."
)

Image features: torch.Size([4491, 768])
Disease prototypes: torch.Size([8, 768])

SUCCESS: Feature and prototype memory are ready for Disease-Aware Attention.


CELL 3 — IMPLEMENT DISEASE-AWARE ATTENTION


Disease-Aware Attention allows each image representation to retrieve
relevant information from the Disease Prototype Memory.

Input:

    image_features
        [B, 768]

Prototype memory:

    disease_prototypes
        [8, 768]

The module learns three projections:

    Query:
        image feature → attention query

    Key:
        disease prototype → disease key

    Value:
        disease prototype → disease value

Attention weights are calculated across all 8 disease prototypes.

The resulting disease context is combined with the original image feature
through a residual connection.

Therefore:

    Image Feature
          +
    Disease Prototype Context
          ↓
    Disease-Aware Feature [B, 768]

The prototype memory is initially treated as fixed memory.

The trainable components are the Query, Key, Value and output projections.

---

In [4]:
# ============================================================
# CELL 3 — IMPLEMENT DISEASE-AWARE ATTENTION
# ============================================================

class DiseaseAwareAttention(nn.Module):

    def __init__(
        self,
        feature_dim=768,
        attention_dim=256,
        num_diseases=8,
        dropout=0.1
    ):
        super().__init__()

        # ----------------------------------------------------
        # QUERY
        # Image Feature → Query
        # ----------------------------------------------------

        self.query_projection = nn.Linear(
            feature_dim,
            attention_dim
        )

        # ----------------------------------------------------
        # KEY
        # Disease Prototype → Key
        # ----------------------------------------------------

        self.key_projection = nn.Linear(
            feature_dim,
            attention_dim
        )

        # ----------------------------------------------------
        # VALUE
        # Disease Prototype → Value
        # ----------------------------------------------------

        self.value_projection = nn.Linear(
            feature_dim,
            attention_dim
        )

        # ----------------------------------------------------
        # DISEASE-SPECIFIC LEARNABLE BIAS
        #
        # One learnable bias value per disease prototype.
        #
        # Shape:
        # [8]
        #
        # This is added to the attention score before Softmax.
        # ----------------------------------------------------

        self.disease_bias = nn.Parameter(
            torch.zeros(num_diseases)
        )

        # ----------------------------------------------------
        # PROJECT ATTENTION OUTPUT BACK
        # TO ORIGINAL FEATURE DIMENSION
        # ----------------------------------------------------

        self.output_projection = nn.Linear(
            attention_dim,
            feature_dim
        )

        self.dropout = nn.Dropout(dropout)

        # Residual normalization
        self.norm = nn.LayerNorm(
            feature_dim
        )

        # Scaling factor
        self.scale = attention_dim ** 0.5


    def forward(
        self,
        image_features,
        prototype_memory
    ):

        # ----------------------------------------------------
        # QUERY
        #
        # [B, 768] → [B, 256]
        # ----------------------------------------------------

        queries = self.query_projection(
            image_features
        )

        # ----------------------------------------------------
        # KEYS
        #
        # [8, 768] → [8, 256]
        # ----------------------------------------------------

        keys = self.key_projection(
            prototype_memory
        )

        # ----------------------------------------------------
        # VALUES
        #
        # [8, 768] → [8, 256]
        # ----------------------------------------------------

        values = self.value_projection(
            prototype_memory
        )

        # ----------------------------------------------------
        # SCALED DOT-PRODUCT ATTENTION
        #
        # [B, 256] @ [256, 8]
        # → [B, 8]
        # ----------------------------------------------------

        attention_scores = (
            queries @ keys.T
        ) / self.scale

        # ----------------------------------------------------
        # ADD DISEASE-SPECIFIC LEARNABLE BIAS
        #
        # [B, 8] + [8]
        # → [B, 8]
        # ----------------------------------------------------

        attention_scores = (
            attention_scores
            + self.disease_bias
        )

        # ----------------------------------------------------
        # SOFTMAX OVER DISEASE PROTOTYPES
        #
        # [B, 8]
        # ----------------------------------------------------

        attention_weights = F.softmax(
            attention_scores,
            dim=1
        )

        attention_weights = self.dropout(
            attention_weights
        )

        # ----------------------------------------------------
        # WEIGHTED DISEASE PROTOTYPE CONTEXT
        #
        # [B, 8] @ [8, 256]
        # → [B, 256]
        # ----------------------------------------------------

        disease_context = (
            attention_weights
            @ values
        )

        # ----------------------------------------------------
        # PROJECT BACK TO ORIGINAL FEATURE SPACE
        #
        # [B, 256] → [B, 768]
        # ----------------------------------------------------

        disease_context = self.output_projection(
            disease_context
        )

        # ----------------------------------------------------
        # RESIDUAL CONNECTION
        #
        # Original Image Feature
        #           +
        # Disease-Aware Context
        #
        # → Disease-Aware Feature [B, 768]
        # ----------------------------------------------------

        disease_aware_features = self.norm(
            image_features
            + self.dropout(disease_context)
        )

        return (
            disease_aware_features,
            attention_weights
        )


# ============================================================
# INITIALIZE MODULE
# ============================================================

disease_attention = DiseaseAwareAttention(
    feature_dim=FEATURE_DIM,
    attention_dim=256,
    num_diseases=NUM_CLASSES,
    dropout=0.1
).to(device)

print(disease_attention)

print(
    "Disease bias shape:",
    disease_attention.disease_bias.shape
)

print(
    "Initial disease bias:",
    disease_attention.disease_bias.detach().cpu()
)

DiseaseAwareAttention(
  (query_projection): Linear(in_features=768, out_features=256, bias=True)
  (key_projection): Linear(in_features=768, out_features=256, bias=True)
  (value_projection): Linear(in_features=768, out_features=256, bias=True)
  (output_projection): Linear(in_features=256, out_features=768, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
)
Disease bias shape: torch.Size([8])
Initial disease bias: tensor([0., 0., 0., 0., 0., 0., 0., 0.])


 CELL 4 — CONTROLLED FORWARD PASS


We perform the first forward pass through Disease-Aware Attention.

Input:

    FiLM-conditioned image features
    [4491, 768]

Prototype Memory:

    [8, 768]

Expected outputs:

    Disease-Aware Features
    [4491, 768]

    Attention Weights
    [4491, 8]

At this stage, the attention module is not trained yet.

Therefore, this cell only verifies that the mathematical flow of the
Disease-Aware Attention module is structurally correct.

---

In [5]:
# ============================================================
# CELL 4 — CONTROLLED FORWARD PASS
# ============================================================

disease_attention.eval()

with torch.no_grad():

    sample_features = train_features.to(device)
    prototype_memory_device = (
        disease_prototypes.to(device)
    )

    disease_aware_features, attention_weights = (
        disease_attention(
            sample_features,
            prototype_memory_device
        )
    )

print(
    "Input features:",
    sample_features.shape
)

print(
    "Disease-aware features:",
    disease_aware_features.shape
)

print(
    "Attention weights:",
    attention_weights.shape
)

assert disease_aware_features.shape == (
    train_features.shape[0],
    FEATURE_DIM
)

assert attention_weights.shape == (
    train_features.shape[0],
    NUM_CLASSES
)

print(
    "\nSUCCESS: Forward pass completed correctly."
)

Input features: torch.Size([4491, 768])
Disease-aware features: torch.Size([4491, 768])
Attention weights: torch.Size([4491, 8])

SUCCESS: Forward pass completed correctly.


CELL 5 — ATTENTION INTEGRITY CHECK


Disease-Aware Attention produces one attention distribution across the
8 disease prototypes for every eye.

For each eye:

    attention weights
        [w_N, w_D, w_G, w_C, w_A, w_H, w_M, w_O]

must:

    • Contain only finite values
    • Contain no negative values
    • Sum to approximately 1

These checks verify the Softmax attention mechanism.

---

In [6]:
# ============================================================
# CELL 5 — ATTENTION INTEGRITY CHECK
# ============================================================

attention_row_sums = (
    attention_weights.sum(dim=1)
)

attention_checks = {
    "Correct attention shape":
        tuple(attention_weights.shape)
        == (len(train_features), NUM_CLASSES),

    "All values finite":
        torch.isfinite(attention_weights).all().item(),

    "No negative attention values":
        torch.all(attention_weights >= 0).item(),

    "All attention rows sum to 1":
        torch.allclose(
            attention_row_sums,
            torch.ones_like(attention_row_sums),
            atol=1e-6
        )
}

for check, result in attention_checks.items():
    print(
        f"{check}: {'PASS' if result else 'FAIL'}"
    )

assert all(attention_checks.values())

print("\nAttention integrity check PASSED.")

Correct attention shape: PASS
All values finite: PASS
No negative attention values: PASS
All attention rows sum to 1: PASS

Attention integrity check PASSED.


 CELL 6 — SAMPLE DISEASE ATTENTION INSPECTION


We inspect the attention distribution for a small number of sample eyes.

Each attention position corresponds to the fixed disease order:

    N, D, G, C, A, H, M, O

Because the Disease-Aware Attention module has not been trained yet,
these values are not interpreted as diagnostic predictions.

This inspection only confirms that the module produces a valid
disease-wise attention distribution for each eye.

---

In [7]:
# ============================================================
# CELL 6 — SAMPLE DISEASE ATTENTION INSPECTION
# ============================================================

num_samples_to_inspect = 5

sample_attention = (
    attention_weights[
        :num_samples_to_inspect
    ]
    .detach()
    .cpu()
)

sample_labels = (
    train_labels[
        :num_samples_to_inspect
    ]
    .cpu()
)

for i in range(num_samples_to_inspect):

    print("\n" + "=" * 70)
    print(f"Sample Eye {i}")

    true_diseases = [
        LABEL_COLUMNS[j]
        for j in range(NUM_CLASSES)
        if sample_labels[i, j] == 1
    ]

    print(
        "True label(s):",
        true_diseases
    )

    print("\nAttention over prototypes:")

    for disease, weight in zip(
        LABEL_COLUMNS,
        sample_attention[i]
    ):
        print(
            f"{disease}: {weight.item():.4f}"
        )


Sample Eye 0
True label(s): ['N']

Attention over prototypes:
N: 0.1250
D: 0.1251
G: 0.1250
C: 0.1250
A: 0.1249
H: 0.1252
M: 0.1248
O: 0.1250

Sample Eye 1
True label(s): ['N']

Attention over prototypes:
N: 0.1251
D: 0.1251
G: 0.1250
C: 0.1249
A: 0.1249
H: 0.1252
M: 0.1248
O: 0.1250

Sample Eye 2
True label(s): ['D', 'O']

Attention over prototypes:
N: 0.1250
D: 0.1250
G: 0.1251
C: 0.1251
A: 0.1249
H: 0.1250
M: 0.1248
O: 0.1250

Sample Eye 3
True label(s): ['O']

Attention over prototypes:
N: 0.1251
D: 0.1251
G: 0.1250
C: 0.1251
A: 0.1249
H: 0.1252
M: 0.1246
O: 0.1250

Sample Eye 4
True label(s): ['D', 'O']

Attention over prototypes:
N: 0.1251
D: 0.1252
G: 0.1250
C: 0.1249
A: 0.1249
H: 0.1252
M: 0.1248
O: 0.1251


CELL 7 — GRADIENT FLOW VERIFICATION


Before training Disease-Aware Attention, we verify that gradients flow
through all learnable components:

    • Query projection
    • Key projection
    • Value projection
    • Disease-specific bias

This confirms that every trainable component can receive updates during
optimization.

---

In [8]:
# ============================================================
# CELL 7 — GRADIENT FLOW VERIFICATION
# ============================================================

disease_attention.train()

# Clear any existing gradients
disease_attention.zero_grad()

# Use a small batch for the gradient test
gradient_test_features = (
    train_features[:32]
    .to(device)
)

gradient_test_prototypes = (
    disease_prototypes
    .to(device)
)

gradient_output, gradient_attention = (
    disease_attention(
        gradient_test_features,
        gradient_test_prototypes
    )
)

# Simple scalar objective only for gradient verification
gradient_test_loss = gradient_output.mean()

gradient_test_loss.backward()

gradient_checks = {
    "Query projection gradient":
        disease_attention.query_projection.weight.grad is not None,

    "Key projection gradient":
        disease_attention.key_projection.weight.grad is not None,

    "Value projection gradient":
        disease_attention.value_projection.weight.grad is not None,

    "Disease bias gradient":
        disease_attention.disease_bias.grad is not None
}

for check, result in gradient_checks.items():
    print(
        f"{check}: {'PASS' if result else 'FAIL'}"
    )

assert all(gradient_checks.values())

print("\nSUCCESS: Gradients flow through all learnable components.")

# Clear temporary gradients
disease_attention.zero_grad()

Query projection gradient: PASS
Key projection gradient: PASS
Value projection gradient: PASS
Disease bias gradient: PASS

SUCCESS: Gradients flow through all learnable components.


 CELL 8 — DISEASE-AWARE ATTENTION STRUCTURAL CHECK


The Disease-Aware Attention module is now structurally verified.

We confirm:

    Input:
        [B, 768]

    Disease Prototype Memory:
        [8, 768]

    Query:
        Q(F)

    Keys:
        K(P)

    Values:
        V(P)

    Attention:
        Softmax(QKᵀ / √d_k + B_bias)

    Output:
        F_d = Attention × V(P)
        [B, 768]

The module is now ready to be connected to a trainable disease prediction
objective so that Q, K, V and B_bias can learn disease-relevant attention.

---

In [9]:
# ============================================================
# CELL 8 — DISEASE-AWARE ATTENTION STRUCTURAL CHECK
# ============================================================

disease_attention.eval()

with torch.no_grad():

    final_test_features = (
        train_features[:64]
        .to(device)
    )

    final_disease_features, final_attention = (
        disease_attention(
            final_test_features,
            disease_prototypes.to(device)
        )
    )

structural_checks = {
    "Input feature dimension":
        final_test_features.shape[1] == FEATURE_DIM,

    "Prototype memory shape":
        tuple(disease_prototypes.shape)
        == (NUM_CLASSES, FEATURE_DIM),

    "Disease-aware output shape":
        tuple(final_disease_features.shape)
        == (64, FEATURE_DIM),

    "Attention output shape":
        tuple(final_attention.shape)
        == (64, NUM_CLASSES),

    "Attention rows sum to 1":
        torch.allclose(
            final_attention.sum(dim=1),
            torch.ones(
                final_attention.size(0),
                device=device
            ),
            atol=1e-6
        ),

    "Disease bias shape":
        tuple(
            disease_attention.disease_bias.shape
        )
        == (NUM_CLASSES,)
}

print("=" * 70)
print("DISEASE-AWARE ATTENTION STRUCTURAL CHECK")
print("=" * 70)

for check, result in structural_checks.items():
    status = "PASS" if result else "FAIL"
    print(f"{status} — {check}")

assert all(structural_checks.values())

print("\n" + "=" * 70)
print("MODULE STRUCTURALLY VERIFIED")
print("=" * 70)

print(
    "\nNext stage: train Disease-Aware Attention "
    "with a multi-label disease prediction objective."
)

DISEASE-AWARE ATTENTION STRUCTURAL CHECK
PASS — Input feature dimension
PASS — Prototype memory shape
PASS — Disease-aware output shape
PASS — Attention output shape
PASS — Attention rows sum to 1
PASS — Disease bias shape

MODULE STRUCTURALLY VERIFIED

Next stage: train Disease-Aware Attention with a multi-label disease prediction objective.


 CELL 9 — PREPARE TRAIN / VALIDATION SPLIT


Disease-Aware Attention must be trained and evaluated on separate data.

The saved feature set contains:

    4491 FiLM-conditioned image features
    [4491, 768]

with corresponding multi-label disease targets:

    [4491, 8]

We create a training and validation split while preserving the alignment
between each feature vector and its multi-label disease target.

The validation set will be used to monitor whether the newly trained
Disease-Aware Attention module generalizes beyond the features used for
optimization.

---

In [10]:
# ============================================================
# CELL 9 — PREPARE TRAIN / VALIDATION SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

indices = np.arange(len(train_features))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

train_idx = torch.tensor(
    train_idx,
    dtype=torch.long
)

val_idx = torch.tensor(
    val_idx,
    dtype=torch.long
)

daa_train_features = train_features[train_idx]
daa_train_labels = train_labels[train_idx]

daa_val_features = train_features[val_idx]
daa_val_labels = train_labels[val_idx]

print("Total samples:", len(train_features))

print(
    "\nTraining features:",
    daa_train_features.shape
)

print(
    "Validation features:",
    daa_val_features.shape
)

print(
    "\nTraining labels:",
    daa_train_labels.shape
)

print(
    "Validation labels:",
    daa_val_labels.shape
)

assert (
    len(daa_train_features)
    + len(daa_val_features)
    == len(train_features)
)

print("\nSUCCESS: Train/validation split created.")

Total samples: 4491

Training features: torch.Size([3592, 768])
Validation features: torch.Size([899, 768])

Training labels: torch.Size([3592, 8])
Validation labels: torch.Size([899, 8])

SUCCESS: Train/validation split created.


 CELL 10 — CREATE DATASETS AND DATALOADERS


The Disease-Aware Attention module will be trained using mini-batches.

Each batch contains:

    Features:
        [B, 768]

    Multi-label targets:
        [B, 8]

The Disease Prototype Memory remains shared across all batches and will be
provided to the Disease-Aware Attention module during the forward pass.

Only the feature representations and their corresponding disease labels
are stored in the DataLoader.


In [11]:
# ============================================================
# CELL 10 — CREATE DATASETS AND DATALOADERS
# ============================================================

from torch.utils.data import (
    TensorDataset,
    DataLoader
)

BATCH_SIZE = 64

daa_train_dataset = TensorDataset(
    daa_train_features,
    daa_train_labels
)

daa_val_dataset = TensorDataset(
    daa_val_features,
    daa_val_labels
)

daa_train_loader = DataLoader(
    daa_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    pin_memory=True
)

daa_val_loader = DataLoader(
    daa_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    pin_memory=True
)

print(
    "Training batches:",
    len(daa_train_loader)
)

print(
    "Validation batches:",
    len(daa_val_loader)
)

sample_batch_features, sample_batch_labels = (
    next(iter(daa_train_loader))
)

print(
    "\nFeature batch shape:",
    sample_batch_features.shape
)

print(
    "Label batch shape:",
    sample_batch_labels.shape
)

Training batches: 57
Validation batches: 15

Feature batch shape: torch.Size([64, 768])
Label batch shape: torch.Size([64, 8])


 CELL 11 — CALCULATE MULTI-LABEL CLASS WEIGHTS


The disease labels are multi-label and potentially imbalanced.

For BCEWithLogitsLoss, we calculate a positive-class weight for each
disease using only the training split.

For each disease:

    pos_weight =
        negative samples / positive samples

This increases the loss contribution of underrepresented positive disease
labels.

The weights are calculated only from the training data to avoid using
validation label statistics during optimization.

---

In [12]:
# ============================================================
# CELL 11 — CALCULATE MULTI-LABEL CLASS WEIGHTS
# ============================================================

positive_counts = daa_train_labels.sum(dim=0)

negative_counts = (
    len(daa_train_labels)
    - positive_counts
)

pos_weight = (
    negative_counts
    / positive_counts.clamp(min=1)
)

class_weight_df = pd.DataFrame({
    "Disease": LABEL_COLUMNS,
    "Positive Samples":
        positive_counts.numpy().astype(int),
    "Negative Samples":
        negative_counts.numpy().astype(int),
    "pos_weight":
        pos_weight.numpy()
})

display(class_weight_df)

print(
    "\nPositive class weights:",
    pos_weight
)

,Disease,Positive Samples,Negative Samples,pos_weight
0,N,1199,2393,1.995830
1,D,1198,2394,1.998331
2,G,214,3378,15.785047
3,C,232,3360,14.482759
4,A,171,3421,20.005848
5,H,115,3477,30.234783
6,M,183,3409,18.628416
7,O,850,2742,3.225882



Positive class weights: tensor([ 1.9958,  1.9983, 15.7850, 14.4828, 20.0058, 30.2348, 18.6284,  3.2259])


 CELL 12 — DISEASE-AWARE ATTENTION TRAINING MODEL


The Disease-Aware Attention module requires a supervised objective so that
its Query, Key, Value projections and disease-specific bias can learn
disease-relevant attention patterns.

A temporary multi-label classification head is attached to the
Disease-Aware Feature representation.

Training flow:

    FiLM-conditioned Feature
            ↓
    Disease-Aware Attention
            ↓
    Disease-Aware Feature [768]
            ↓
    Multi-Label Classification Head
            ↓
    Disease Logits [8]
            ↓
    BCEWithLogitsLoss

The classification head provides the learning signal required to train the
Disease-Aware Attention module.

The central output of this stage remains the trained Disease-Aware
Attention representation.


---

In [13]:
# ============================================================
# CELL 12 — DISEASE-AWARE ATTENTION TRAINING MODEL
# ============================================================

class DiseaseAwareAttentionClassifier(nn.Module):

    def __init__(
        self,
        attention_module,
        feature_dim=768,
        num_classes=8
    ):
        super().__init__()

        self.attention = attention_module

        self.classifier = nn.Linear(
            feature_dim,
            num_classes
        )


    def forward(
        self,
        image_features,
        prototype_memory
    ):

        disease_features, attention_weights = (
            self.attention(
                image_features,
                prototype_memory
            )
        )

        logits = self.classifier(
            disease_features
        )

        return (
            logits,
            disease_features,
            attention_weights
        )


daa_model = DiseaseAwareAttentionClassifier(
    attention_module=disease_attention,
    feature_dim=FEATURE_DIM,
    num_classes=NUM_CLASSES
).to(device)

print(daa_model)

DiseaseAwareAttentionClassifier(
  (attention): DiseaseAwareAttention(
    (query_projection): Linear(in_features=768, out_features=256, bias=True)
    (key_projection): Linear(in_features=768, out_features=256, bias=True)
    (value_projection): Linear(in_features=768, out_features=256, bias=True)
    (output_projection): Linear(in_features=256, out_features=768, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
    (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (classifier): Linear(in_features=768, out_features=8, bias=True)
)


 CELL 13 — TRAINING CONFIGURATION


The Disease-Aware Attention training objective is multi-label
classification.

We use:

    BCEWithLogitsLoss

because multiple diseases may be present in the same eye.

Positive class weights are included to account for class imbalance.

The optimizer updates:

    • Query projection
    • Key projection
    • Value projection
    • Disease-specific bias
    • Output projection
    • LayerNorm
    • Temporary classification head

The Disease Prototype Memory remains fixed during this initial attention
training stage.

Training will begin with a controlled sanity check before running the full
training loop.

---

In [14]:
# ============================================================
# CELL 13 — TRAINING CONFIGURATION
# ============================================================

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight.to(device)
)

optimizer = torch.optim.AdamW(
    daa_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in daa_model.parameters()
    if parameter.requires_grad
)

print(
    "Loss function:",
    criterion
)

print(
    "\nOptimizer:",
    optimizer.__class__.__name__
)

print(
    "\nLearning rate:",
    LEARNING_RATE
)

print(
    "Weight decay:",
    WEIGHT_DECAY
)

print(
    "\nTrainable parameters:",
    f"{trainable_parameters:,}"
)

print(
    "\nSUCCESS: Training configuration is ready."
)

Loss function: BCEWithLogitsLoss()

Optimizer: AdamW

Learning rate: 0.001
Weight decay: 0.0001

Trainable parameters: 795,664

SUCCESS: Training configuration is ready.


 CELL 14 — SINGLE-BATCH TRAINING SANITY CHECK


Before running full training, we perform one controlled optimization step.

This verifies that:

    • The model produces valid logits
    • BCEWithLogitsLoss is finite
    • Backpropagation completes successfully
    • The optimizer can update the trainable parameters

This is only a technical sanity check.

The full training process begins in the following cells.


---

In [15]:
# ============================================================
# CELL 14 — SINGLE-BATCH TRAINING SANITY CHECK
# ============================================================

daa_model.train()

sanity_features, sanity_labels = next(
    iter(daa_train_loader)
)

sanity_features = sanity_features.to(
    device,
    non_blocking=True
)

sanity_labels = sanity_labels.to(
    device,
    non_blocking=True
)

optimizer.zero_grad()

sanity_logits, sanity_features_out, sanity_attention = (
    daa_model(
        sanity_features,
        disease_prototypes.to(device)
    )
)

sanity_loss = criterion(
    sanity_logits,
    sanity_labels
)

assert torch.isfinite(sanity_loss)

sanity_loss.backward()

optimizer.step()

print(
    "Logits shape:",
    sanity_logits.shape
)

print(
    "Disease-aware feature shape:",
    sanity_features_out.shape
)

print(
    "Attention shape:",
    sanity_attention.shape
)

print(
    f"\nSanity-check loss: "
    f"{sanity_loss.item():.6f}"
)

print(
    "\nSUCCESS: One training step completed."
)

Logits shape: torch.Size([64, 8])
Disease-aware feature shape: torch.Size([64, 768])
Attention shape: torch.Size([64, 8])

Sanity-check loss: 1.123076

SUCCESS: One training step completed.


CELL 15 — PARAMETER UPDATE VERIFICATION


A successful backward pass is not enough by itself.

We verify that an optimizer step actually changes the parameters of the
Disease-Aware Attention model.

We compare the Query projection weights before and after one optimization
step.

A non-zero change confirms that training updates are being applied.

---

In [16]:
# ============================================================
# CELL 15 — PARAMETER UPDATE VERIFICATION
# ============================================================

daa_model.train()

update_features, update_labels = next(
    iter(daa_train_loader)
)

update_features = update_features.to(
    device,
    non_blocking=True
)

update_labels = update_labels.to(
    device,
    non_blocking=True
)

# Save parameter before update
query_weight_before = (
    daa_model
    .attention
    .query_projection
    .weight
    .detach()
    .clone()
)

optimizer.zero_grad()

update_logits, _, _ = daa_model(
    update_features,
    disease_prototypes.to(device)
)

update_loss = criterion(
    update_logits,
    update_labels
)

update_loss.backward()

optimizer.step()

# Save parameter after update
query_weight_after = (
    daa_model
    .attention
    .query_projection
    .weight
    .detach()
    .clone()
)

parameter_change = torch.norm(
    query_weight_after
    - query_weight_before
).item()

print(
    f"Training loss: {update_loss.item():.6f}"
)

print(
    f"Query projection parameter change: "
    f"{parameter_change:.8f}"
)

assert parameter_change > 0

print(
    "\nSUCCESS: Model parameters are updating."
)

Training loss: 1.791040
Query projection parameter change: 0.31690869

SUCCESS: Model parameters are updating.


 CELL 16 — TRAINING AND VALIDATION FUNCTIONS


We define separate functions for training and validation.

Training:

    • Model in training mode
    • Gradients enabled
    • Parameters updated

Validation:

    • Model in evaluation mode
    • No gradients
    • No parameter updates

Both functions return the average BCEWithLogitsLoss across all batches.

The prototype memory remains fixed and is shared across all batches.

---

In [17]:
# ============================================================
# CELL 16 — TRAINING AND VALIDATION FUNCTIONS
# ============================================================

prototype_memory_device = (
    disease_prototypes
    .to(device)
)


def train_one_epoch(
    model,
    data_loader,
    criterion,
    optimizer,
    prototype_memory,
    device
):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for batch_features, batch_labels in data_loader:

        batch_features = batch_features.to(
            device,
            non_blocking=True
        )

        batch_labels = batch_labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad()

        logits, _, _ = model(
            batch_features,
            prototype_memory
        )

        loss = criterion(
            logits,
            batch_labels
        )

        loss.backward()

        optimizer.step()

        batch_size = batch_features.size(0)

        total_loss += (
            loss.item()
            * batch_size
        )

        total_samples += batch_size

    return total_loss / total_samples


def validate_one_epoch(
    model,
    data_loader,
    criterion,
    prototype_memory,
    device
):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    with torch.no_grad():

        for batch_features, batch_labels in data_loader:

            batch_features = batch_features.to(
                device,
                non_blocking=True
            )

            batch_labels = batch_labels.to(
                device,
                non_blocking=True
            )

            logits, _, _ = model(
                batch_features,
                prototype_memory
            )

            loss = criterion(
                logits,
                batch_labels
            )

            batch_size = batch_features.size(0)

            total_loss += (
                loss.item()
                * batch_size
            )

            total_samples += batch_size

    return total_loss / total_samples


print(
    "Training and validation functions are ready."
)

Training and validation functions are ready.


 CELL 17 — TRAIN DISEASE-AWARE ATTENTION


We now train the Disease-Aware Attention module using the multi-label
classification objective.

The model learns:

    • Q(F): how each image queries disease information
    • K(P): how disease prototypes are represented as attention keys
    • V(P): what disease information is retrieved
    • B_bias: disease-specific attention adjustment
    • Output projection and normalization
    • Temporary multi-label classification mapping

The Disease Prototype Memory itself remains fixed.

Training and validation loss are recorded after every epoch.

The best model is selected using the lowest validation loss.

---

In [18]:
# ============================================================
# CELL 17 — TRAIN DISEASE-AWARE ATTENTION
# ============================================================

NUM_EPOCHS = 20

best_val_loss = float("inf")

training_history = {
    "train_loss": [],
    "val_loss": []
}

best_model_state = None


for epoch in range(NUM_EPOCHS):

    train_loss = train_one_epoch(
        model=daa_model,
        data_loader=daa_train_loader,
        criterion=criterion,
        optimizer=optimizer,
        prototype_memory=prototype_memory_device,
        device=device
    )

    val_loss = validate_one_epoch(
        model=daa_model,
        data_loader=daa_val_loader,
        criterion=criterion,
        prototype_memory=prototype_memory_device,
        device=device
    )

    training_history["train_loss"].append(
        train_loss
    )

    training_history["val_loss"].append(
        val_loss
    )

    print(
        f"Epoch [{epoch + 1:02d}/{NUM_EPOCHS}] "
        f"| Train Loss: {train_loss:.6f} "
        f"| Val Loss: {val_loss:.6f}"
    )

    # Save best state in memory
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_model_state = {
            name: parameter.detach()
            .cpu()
            .clone()

            for name, parameter in
            daa_model.state_dict().items()
        }

        print(
            "   ✓ Best validation model updated"
        )


print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(
    f"Best Validation Loss: "
    f"{best_val_loss:.6f}"
)

Epoch [01/20] | Train Loss: 1.068911 | Val Loss: 0.981864
   ✓ Best validation model updated
Epoch [02/20] | Train Loss: 0.931504 | Val Loss: 0.926585
   ✓ Best validation model updated
Epoch [03/20] | Train Loss: 0.897429 | Val Loss: 0.902710
   ✓ Best validation model updated
Epoch [04/20] | Train Loss: 0.901312 | Val Loss: 0.953480
Epoch [05/20] | Train Loss: 0.882964 | Val Loss: 0.873038
   ✓ Best validation model updated
Epoch [06/20] | Train Loss: 0.855382 | Val Loss: 0.902978
Epoch [07/20] | Train Loss: 0.833478 | Val Loss: 0.966787
Epoch [08/20] | Train Loss: 0.861913 | Val Loss: 0.934673
Epoch [09/20] | Train Loss: 0.836419 | Val Loss: 0.933383
Epoch [10/20] | Train Loss: 0.823819 | Val Loss: 0.924152
Epoch [11/20] | Train Loss: 0.824554 | Val Loss: 0.894296
Epoch [12/20] | Train Loss: 0.797253 | Val Loss: 0.852836
   ✓ Best validation model updated
Epoch [13/20] | Train Loss: 0.818313 | Val Loss: 0.911621
Epoch [14/20] | Train Loss: 0.795147 | Val Loss: 0.836883
   ✓ Best val

CELL 18 — SAVE BEST DISEASE-AWARE ATTENTION MODEL


After training, we save the best model based on validation loss.

The checkpoint contains:

    • Complete trained model state
    • Disease-Aware Attention parameters
    • Temporary classifier parameters
    • Best validation loss
    • Training history
    • Disease label ordering
    • Feature and prototype dimensions

The Disease Prototype Memory is not duplicated inside this checkpoint.
It remains stored separately as:

    disease_prototypes.pt

Together, the prototype memory and the trained Disease-Aware Attention
checkpoint provide the learned components required for the next stages
of the QCDP-BiFormer pipeline.

---

In [19]:
# ============================================================
# CELL 18 — SAVE BEST DISEASE-AWARE ATTENTION MODEL
# ============================================================

BEST_DAA_MODEL_PATH = os.path.join(
    ROOT,
    "disease_aware_attention_best.pt"
)

checkpoint = {
    "model_state_dict": best_model_state,

    "best_val_loss": best_val_loss,

    "training_history": training_history,

    "label_columns": LABEL_COLUMNS,

    "feature_dim": FEATURE_DIM,

    "attention_dim": 256,

    "num_diseases": NUM_CLASSES,

    "prototype_shape":
        tuple(disease_prototypes.shape),

    "architecture": {
        "query": "768 -> 256",
        "key": "768 -> 256",
        "value": "768 -> 256",
        "output": "256 -> 768",
        "disease_bias": True,
        "residual_connection": True,
        "layer_norm": True
    }
}

torch.save(
    checkpoint,
    BEST_DAA_MODEL_PATH
)

assert os.path.exists(
    BEST_DAA_MODEL_PATH
)

print(
    "Best Disease-Aware Attention model saved."
)

print(
    "Path:",
    BEST_DAA_MODEL_PATH
)

print(
    "Best Validation Loss:",
    best_val_loss
)

Best Disease-Aware Attention model saved.
Path: /content/drive/My Drive/Eye Disease/Dataset/disease_aware_attention_best.pt
Best Validation Loss: 0.836882654299858


 CELL 19 — RELOAD BEST DISEASE-AWARE ATTENTION MODEL


During training, the best model was selected using the lowest validation
loss.

Best result:

    Epoch 18
    Best Validation Loss: 0.823291

We now reload this saved checkpoint so that all subsequent feature
generation uses the best learned Disease-Aware Attention parameters,
rather than the parameters remaining after the final training epoch.

The checkpoint contains:

    • Trained Disease-Aware Attention parameters
    • Temporary classifier parameters
    • Disease-specific learnable bias
    • Training history
    • Architecture metadata

The trained model will be used to generate the final Disease-Aware
Feature representations for the complete feature set.

---

In [20]:
# ============================================================
# CELL 19 — RELOAD BEST DISEASE-AWARE ATTENTION MODEL
# ============================================================

checkpoint = torch.load(
    BEST_DAA_MODEL_PATH,
    map_location=device,
    weights_only=False
)

print("Checkpoint loaded successfully.")

print(
    "\nBest validation loss:",
    checkpoint["best_val_loss"]
)

print(
    "Feature dimension:",
    checkpoint["feature_dim"]
)

print(
    "Attention dimension:",
    checkpoint["attention_dim"]
)

print(
    "Number of diseases:",
    checkpoint["num_diseases"]
)

print(
    "\nLabel order:",
    checkpoint["label_columns"]
)

Checkpoint loaded successfully.

Best validation loss: 0.836882654299858
Feature dimension: 768
Attention dimension: 256
Number of diseases: 8

Label order: ['N', 'D', 'G', 'C', 'A', 'H', 'M', 'O']


 CELL 20 — RESTORE AND VERIFY BEST MODEL


We recreate the Disease-Aware Attention training model and load the
saved best parameters.

After restoration, we verify:

    • All expected parameters are present
    • The model loads without missing or unexpected keys
    • The model is placed in evaluation mode

The temporary classifier remains attached only because it is part of the
saved training checkpoint.

For the next pipeline stage, our main interest is the trained:

    Disease-Aware Attention module

and its resulting Disease-Aware Feature representation.

---

In [21]:
# ============================================================
# CELL 20 — RESTORE AND VERIFY BEST MODEL
# ============================================================

# Recreate the model architecture
best_daa_model = DiseaseAwareAttentionClassifier(
    attention_module=DiseaseAwareAttention(
        feature_dim=FEATURE_DIM,
        attention_dim=256,
        num_diseases=NUM_CLASSES,
        dropout=0.1
    ),
    feature_dim=FEATURE_DIM,
    num_classes=NUM_CLASSES
).to(device)

# Load best trained parameters
load_result = best_daa_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

best_daa_model.eval()

print(
    "Missing keys:",
    load_result.missing_keys
)

print(
    "Unexpected keys:",
    load_result.unexpected_keys
)

assert len(load_result.missing_keys) == 0
assert len(load_result.unexpected_keys) == 0

print("\nSUCCESS: Best model restored correctly.")

Missing keys: []
Unexpected keys: []

SUCCESS: Best model restored correctly.


 CELL 21 — GENERATE FINAL DISEASE-AWARE FEATURES


We now pass all 4491 FiLM-conditioned image features through the
best trained Disease-Aware Attention module.

Input:

    FiLM-conditioned features
    [4491, 768]

Disease Prototype Memory:

    [8, 768]

Output:

    Disease-Aware Features
    [4491, 768]

    Disease Attention Weights
    [4491, 8]

These representations are generated using the best validation model and
will serve as the learned output of the Disease-Aware Attention stage.

The feature order is preserved exactly so that each generated feature
remains aligned with its original sample and multi-label target.

---

In [22]:
# ============================================================
# CELL 21 — GENERATE FINAL DISEASE-AWARE FEATURES
# ============================================================

best_attention_module = best_daa_model.attention

best_attention_module.eval()

all_disease_aware_features = []
all_trained_attention_weights = []

feature_generation_loader = DataLoader(
    train_features,
    batch_size=128,
    shuffle=False,
    pin_memory=True
)

prototype_memory_device = (
    disease_prototypes
    .to(device)
)

with torch.no_grad():

    for batch_features in feature_generation_loader:

        batch_features = batch_features.to(
            device,
            non_blocking=True
        )

        batch_disease_features, batch_attention = (
            best_attention_module(
                batch_features,
                prototype_memory_device
            )
        )

        all_disease_aware_features.append(
            batch_disease_features.cpu()
        )

        all_trained_attention_weights.append(
            batch_attention.cpu()
        )


all_disease_aware_features = torch.cat(
    all_disease_aware_features,
    dim=0
)

all_trained_attention_weights = torch.cat(
    all_trained_attention_weights,
    dim=0
)

print(
    "Final disease-aware features:",
    all_disease_aware_features.shape
)

print(
    "Final attention weights:",
    all_trained_attention_weights.shape
)

assert tuple(all_disease_aware_features.shape) == (
    len(train_features),
    FEATURE_DIM
)

assert tuple(all_trained_attention_weights.shape) == (
    len(train_features),
    NUM_CLASSES
)

print(
    "\nSUCCESS: Final Disease-Aware features generated."
)

Final disease-aware features: torch.Size([4491, 768])
Final attention weights: torch.Size([4491, 8])

SUCCESS: Final Disease-Aware features generated.


 CELL 22 — INSPECT TRAINED DISEASE ATTENTION


Before training, Disease-Aware Attention produced an almost uniform
distribution across the 8 disease prototypes:

    approximately 0.125 per prototype

After supervised training, we inspect the learned attention distributions.

For each sample, attention values still represent prototype relevance and
must not be interpreted directly as final disease probabilities.

The purpose of this inspection is to verify that attention has become
input-dependent and is no longer uniformly distributed across all
prototypes.

---

In [23]:
# ============================================================
# CELL 22 — INSPECT TRAINED DISEASE ATTENTION
# ============================================================

num_samples_to_inspect = 5

for i in range(num_samples_to_inspect):

    print("\n" + "=" * 70)
    print(f"Sample Eye {i}")

    true_diseases = [
        LABEL_COLUMNS[j]
        for j in range(NUM_CLASSES)
        if train_labels[i, j] == 1
    ]

    print(
        "True label(s):",
        true_diseases
    )

    print("\nTrained attention over prototypes:")

    for disease, weight in zip(
        LABEL_COLUMNS,
        all_trained_attention_weights[i]
    ):
        print(
            f"{disease}: {weight.item():.4f}"
        )

    print(
        "\nMaximum attended prototype:",
        LABEL_COLUMNS[
            torch.argmax(
                all_trained_attention_weights[i]
            ).item()
        ]
    )


Sample Eye 0
True label(s): ['N']

Trained attention over prototypes:
N: 0.0052
D: 0.0042
G: 0.0351
C: 0.9057
A: 0.0057
H: 0.0033
M: 0.0331
O: 0.0076

Maximum attended prototype: C

Sample Eye 1
True label(s): ['N']

Trained attention over prototypes:
N: 0.0081
D: 0.0067
G: 0.0465
C: 0.8686
A: 0.0088
H: 0.0053
M: 0.0446
O: 0.0113

Maximum attended prototype: C

Sample Eye 2
True label(s): ['D', 'O']

Trained attention over prototypes:
N: 0.0222
D: 0.0194
G: 0.0818
C: 0.7326
A: 0.0233
H: 0.0163
M: 0.0758
O: 0.0285

Maximum attended prototype: C

Sample Eye 3
True label(s): ['O']

Trained attention over prototypes:
N: 0.0485
D: 0.0441
G: 0.1274
C: 0.5056
A: 0.0507
H: 0.0391
M: 0.1272
O: 0.0575

Maximum attended prototype: C

Sample Eye 4
True label(s): ['D', 'O']

Trained attention over prototypes:
N: 0.0429
D: 0.0388
G: 0.1186
C: 0.5547
A: 0.0447
H: 0.0340
M: 0.1148
O: 0.0515

Maximum attended prototype: C


 CELL 23 — SAVE DISEASE-AWARE FEATURE MEMORY


The trained Disease-Aware Attention module has now transformed the
FiLM-conditioned feature memory into Disease-Aware representations.

We perform a final integrity check to verify:

    • Correct number of samples
    • Correct feature dimension
    • Correct attention dimension
    • All values are finite
    • Every attention distribution sums to approximately 1

The resulting memory is saved for the next module.

Saved artifacts:

    disease_aware_features.pt
    disease_attention_weights.pt

The feature order is preserved from train_features.pt so that sample
alignment with labels and paired-eye information remains unchanged.

The next stage can therefore begin from:

    disease_aware_features.pt
                +
    disease_attention_weights.pt
                +
    paired-eye information
                ↓
    Cross-Eye Bilateral Attention

---

In [24]:
# ============================================================
# CELL 23 — FINAL CHECK AND SAVE DISEASE-AWARE MEMORY
# ============================================================

# ------------------------------------------------------------
# FINAL INTEGRITY CHECKS
# ------------------------------------------------------------

final_checks = {

    "Correct feature shape":
        tuple(all_disease_aware_features.shape)
        == (len(train_features), FEATURE_DIM),

    "Correct attention shape":
        tuple(all_trained_attention_weights.shape)
        == (len(train_features), NUM_CLASSES),

    "All disease-aware features finite":
        torch.isfinite(
            all_disease_aware_features
        ).all().item(),

    "All attention weights finite":
        torch.isfinite(
            all_trained_attention_weights
        ).all().item(),

    "No negative attention weights":
        torch.all(
            all_trained_attention_weights >= 0
        ).item(),

    "Attention rows sum to 1":
        torch.allclose(
            all_trained_attention_weights.sum(dim=1),
            torch.ones(
                len(all_trained_attention_weights)
            ),
            atol=1e-6
        )
}


print("=" * 70)
print("FINAL DISEASE-AWARE MEMORY CHECK")
print("=" * 70)

for check, result in final_checks.items():

    print(
        f"{'PASS' if result else 'FAIL'} — {check}"
    )

assert all(final_checks.values())


# ------------------------------------------------------------
# SAVE TRAINED FEATURE MEMORY
# ------------------------------------------------------------

DISEASE_AWARE_FEATURES_PATH = os.path.join(
    ROOT,
    "disease_aware_features.pt"
)

DISEASE_ATTENTION_WEIGHTS_PATH = os.path.join(
    ROOT,
    "disease_attention_weights.pt"
)


torch.save(
    all_disease_aware_features,
    DISEASE_AWARE_FEATURES_PATH
)

torch.save(
    all_trained_attention_weights,
    DISEASE_ATTENTION_WEIGHTS_PATH
)


assert os.path.exists(
    DISEASE_AWARE_FEATURES_PATH
)

assert os.path.exists(
    DISEASE_ATTENTION_WEIGHTS_PATH
)


print("\n" + "=" * 70)
print("DISEASE-AWARE ATTENTION MODULE COMPLETE")
print("=" * 70)

print(
    "\nSaved feature memory:",
    DISEASE_AWARE_FEATURES_PATH
)

print(
    "Saved attention weights:",
    DISEASE_ATTENTION_WEIGHTS_PATH
)

print(
    "\nFinal feature shape:",
    all_disease_aware_features.shape
)

print(
    "Final attention shape:",
    all_trained_attention_weights.shape
)

FINAL DISEASE-AWARE MEMORY CHECK
PASS — Correct feature shape
PASS — Correct attention shape
PASS — All disease-aware features finite
PASS — All attention weights finite
PASS — No negative attention weights
PASS — Attention rows sum to 1

DISEASE-AWARE ATTENTION MODULE COMPLETE

Saved feature memory: /content/drive/My Drive/Eye Disease/Dataset/disease_aware_features.pt
Saved attention weights: /content/drive/My Drive/Eye Disease/Dataset/disease_attention_weights.pt

Final feature shape: torch.Size([4491, 768])
Final attention shape: torch.Size([4491, 8])


CELL 24 — AVERAGE ATTENTION PER DISEASE PROTOTYPE


The sample inspection suggested that attention may be concentrating heavily
on prototype C.

We now calculate the average attention assigned to each disease prototype
across all 4491 samples.

This provides a global view of prototype usage.

A healthy disease-aware attention mechanism does not necessarily require
equal attention across all prototypes. However, extreme dominance by one
prototype may indicate attention collapse.

---

In [25]:
# ============================================================
# CELL 24 — AVERAGE ATTENTION PER DISEASE PROTOTYPE
# ============================================================

average_attention = (
    all_trained_attention_weights
    .mean(dim=0)
)

print("=" * 70)
print("AVERAGE ATTENTION ACROSS ALL SAMPLES")
print("=" * 70)

for disease, attention in zip(
    LABEL_COLUMNS,
    average_attention
):
    print(
        f"{disease}: {attention.item():.6f}"
    )

print("\nTotal:", average_attention.sum().item())

assert torch.allclose(
    average_attention.sum(),
    torch.tensor(1.0),
    atol=1e-6
)

AVERAGE ATTENTION ACROSS ALL SAMPLES
N: 0.035094
D: 0.032640
G: 0.087138
C: 0.653304
A: 0.036357
H: 0.029686
M: 0.085318
O: 0.040464

Total: 1.0000001192092896


 CELL 25 — PROTOTYPE DOMINANCE FREQUENCY

For every sample, we identify the disease prototype with the highest
attention weight.

We then count how frequently each prototype receives maximum attention
across all 4491 samples.

This helps determine whether different samples dynamically select
different disease prototypes or whether one prototype dominates most
of the dataset.

---

In [26]:
# ============================================================
# CELL 25 — PROTOTYPE DOMINANCE FREQUENCY
# ============================================================

dominant_prototypes = torch.argmax(
    all_trained_attention_weights,
    dim=1
)

dominant_counts = torch.bincount(
    dominant_prototypes,
    minlength=NUM_CLASSES
)

dominant_percentages = (
    dominant_counts.float()
    / len(all_trained_attention_weights)
    * 100
)

print("=" * 70)
print("MAXIMUM-ATTENTION PROTOTYPE FREQUENCY")
print("=" * 70)

for i, disease in enumerate(LABEL_COLUMNS):

    print(
        f"{disease}: "
        f"{dominant_counts[i].item():4d} samples "
        f"({dominant_percentages[i].item():6.2f}%)"
    )

print(
    "\nTotal samples:",
    dominant_counts.sum().item()
)

MAXIMUM-ATTENTION PROTOTYPE FREQUENCY
N:    0 samples (  0.00%)
D:    0 samples (  0.00%)
G:    3 samples (  0.07%)
C: 4333 samples ( 96.48%)
A:    0 samples (  0.00%)
H:   50 samples (  1.11%)
M:  105 samples (  2.34%)
O:    0 samples (  0.00%)

Total samples: 4491


 CELL 26 — ATTENTION ENTROPY ANALYSIS


Attention entropy measures how concentrated or distributed the prototype
attention is.

For 8 disease prototypes:

    Maximum entropy:
        log(8) ≈ 2.079

This corresponds approximately to uniform attention:

    [0.125, 0.125, ..., 0.125]

Low entropy indicates concentrated attention, where one or a small number
of prototypes receive most of the attention.

We calculate entropy for every sample and summarize its distribution.

---

In [27]:
# ============================================================
# CELL 26 — ATTENTION ENTROPY ANALYSIS
# ============================================================

epsilon = 1e-12

attention_entropy = -(
    all_trained_attention_weights
    * torch.log(
        all_trained_attention_weights + epsilon
    )
).sum(dim=1)

max_entropy = np.log(NUM_CLASSES)

print("=" * 70)
print("ATTENTION ENTROPY")
print("=" * 70)

print(
    f"Minimum entropy: "
    f"{attention_entropy.min().item():.6f}"
)

print(
    f"Maximum entropy: "
    f"{attention_entropy.max().item():.6f}"
)

print(
    f"Mean entropy: "
    f"{attention_entropy.mean().item():.6f}"
)

print(
    f"Maximum possible entropy: "
    f"{max_entropy:.6f}"
)

print(
    f"\nNormalized mean entropy: "
    f"{(attention_entropy.mean() / max_entropy).item():.4f}"
)

ATTENTION ENTROPY
Minimum entropy: 0.031511
Maximum entropy: 2.077307
Mean entropy: 1.133772
Maximum possible entropy: 2.079442

Normalized mean entropy: 0.5452


 CELL 27 — ATTENTION PATTERNS BY TRUE DISEASE LABEL


We compare the average attention distribution for samples containing each
disease label.

For every disease:

    • Select all samples where that disease is present
    • Calculate the mean attention over all 8 prototypes

This does not treat attention weights as disease probabilities.

Instead, it helps determine whether the learned Disease-Aware Attention
produces different prototype-attention patterns for different disease
groups.


In [28]:
# ============================================================
# CELL 27 — ATTENTION PATTERNS BY TRUE DISEASE LABEL
# ============================================================

print("=" * 90)
print("AVERAGE ATTENTION FOR SAMPLES CONTAINING EACH DISEASE")
print("=" * 90)

label_attention_profiles = {}

for disease_index, disease_name in enumerate(
    LABEL_COLUMNS
):

    disease_mask = (
        train_labels[:, disease_index] == 1
    )

    disease_attention_profile = (
        all_trained_attention_weights[
            disease_mask
        ]
        .mean(dim=0)
    )

    label_attention_profiles[
        disease_name
    ] = disease_attention_profile

    dominant_disease = LABEL_COLUMNS[
        torch.argmax(
            disease_attention_profile
        ).item()
    ]

    print(
        f"\nTrue Disease Group: {disease_name}"
    )

    print(
        f"Number of samples: "
        f"{disease_mask.sum().item()}"
    )

    print(
        f"Dominant attended prototype: "
        f"{dominant_disease}"
    )

    print(
        "Average attention:"
    )

    for prototype_name, weight in zip(
        LABEL_COLUMNS,
        disease_attention_profile
    ):

        print(
            f"  {prototype_name}: "
            f"{weight.item():.4f}"
        )

AVERAGE ATTENTION FOR SAMPLES CONTAINING EACH DISEASE

True Disease Group: N
Number of samples: 1502
Dominant attended prototype: C
Average attention:
  N: 0.0425
  D: 0.0398
  G: 0.0985
  C: 0.5930
  A: 0.0439
  H: 0.0366
  M: 0.0975
  O: 0.0483

True Disease Group: D
Number of samples: 1491
Dominant attended prototype: C
Average attention:
  N: 0.0249
  D: 0.0225
  G: 0.0753
  C: 0.7284
  A: 0.0260
  H: 0.0199
  M: 0.0730
  O: 0.0300

True Disease Group: G
Number of samples: 269
Dominant attended prototype: C
Average attention:
  N: 0.0745
  D: 0.0723
  G: 0.1255
  C: 0.3783
  A: 0.0759
  H: 0.0688
  M: 0.1250
  O: 0.0797

True Disease Group: C
Number of samples: 286
Dominant attended prototype: C
Average attention:
  N: 0.0562
  D: 0.0534
  G: 0.1146
  C: 0.4933
  A: 0.0579
  H: 0.0492
  M: 0.1124
  O: 0.0630

True Disease Group: A
Number of samples: 215
Dominant attended prototype: C
Average attention:
  N: 0.0183
  D: 0.0162
  G: 0.0624
  C: 0.7878
  A: 0.0192
  H: 0.0140
  M: 0.0

CELL 28 — ATTENTION BEHAVIOR SUMMARY


We summarize the global behavior of the trained Disease-Aware Attention
mechanism.

The summary combines:

    • Most dominant prototype
    • Percentage of samples dominated by that prototype
    • Average attention assigned to that prototype
    • Mean normalized attention entropy

These measurements will guide the next decision:

    A) Attention is sufficiently diverse
       → Keep the current module

    B) Attention shows severe prototype collapse
       → Apply a small training stabilization enhancement and retrain

The proposed Disease-Aware Attention architecture itself is not changed.
Only the training objective or attention stabilization strategy would be
adjusted if required.

---

In [29]:
# ============================================================
# CELL 28 — ATTENTION BEHAVIOR SUMMARY
# ============================================================

most_dominant_index = torch.argmax(
    dominant_counts
).item()

most_dominant_name = (
    LABEL_COLUMNS[most_dominant_index]
)

most_dominant_percentage = (
    dominant_percentages[
        most_dominant_index
    ].item()
)

most_dominant_average_attention = (
    average_attention[
        most_dominant_index
    ].item()
)

normalized_mean_entropy = (
    attention_entropy.mean()
    / max_entropy
).item()


print("=" * 70)
print("DISEASE-AWARE ATTENTION BEHAVIOR SUMMARY")
print("=" * 70)

print(
    f"Most dominant prototype: "
    f"{most_dominant_name}"
)

print(
    f"Dominant for: "
    f"{most_dominant_percentage:.2f}% "
    f"of all samples"
)

print(
    f"Average attention received: "
    f"{most_dominant_average_attention:.6f}"
)

print(
    f"Normalized mean entropy: "
    f"{normalized_mean_entropy:.4f}"
)


print("\nInterpretation:")

if most_dominant_percentage > 80:

    print(
        "WARNING — Strong prototype dominance detected."
    )

elif most_dominant_percentage > 50:

    print(
        "NOTICE — Moderate prototype dominance detected."
    )

else:

    print(
        "GOOD — Attention dominance is distributed "
        "across multiple prototypes."
    )


if normalized_mean_entropy < 0.30:

    print(
        "WARNING — Attention distributions are highly "
        "concentrated."
    )

elif normalized_mean_entropy < 0.60:

    print(
        "NOTICE — Attention distributions show moderate "
        "concentration."
    )

else:

    print(
        "GOOD — Attention distributions retain substantial "
        "diversity."
    )

DISEASE-AWARE ATTENTION BEHAVIOR SUMMARY
Most dominant prototype: C
Dominant for: 96.48% of all samples
Average attention received: 0.653304
Normalized mean entropy: 0.5452

Interpretation:
WARNING — Strong prototype dominance detected.
NOTICE — Attention distributions show moderate concentration.
